# A Motivating Example: the Bigram Language Model

Open in [Google Colab](https://colab.research.google.com/github/febse/ta2025/blob/main/03-02-ANN-Bigram-Motivation.ipynb)

Until now we have treated text as a *bag of words*: the representations we used (counts, TF-IDF) only looked at *which* words appear in a document, but not at *where* they appear or in *what order*. In this chapter we take the first step towards models that use word order. We start with a very simple probabilistic model of language, the **bigram model**, and use it as a running example that motivates neural networks and, later in the course, word embeddings.

## Why is such a simple model interesting?

The question that the bigram model answers is: *given the current word, how likely is each possible next word?* Even this tiny question is surprisingly powerful, because

- it lets us **score** sentences by plausibility: a sentence that a speaker of the language could plausibly say gets a higher probability than a word salad (useful, for example, in speech recognition, machine translation and spelling correction);
- it lets us **generate** text by repeatedly sampling the next word;
- it is a **prediction problem with many classes** (one per word in the vocabulary). Training a model to predict the next word is exactly the task on which modern language models are built, so it is the simplest member of a long family of models;
- its weakness (it needs a huge table of probabilities that cannot be estimated reliably and treats all words as unrelated) is precisely what neural networks and word embeddings fix.


## The Bi-Gram Model

Until now we exclusively looked into bag-of-words models that ignore the order of tokens in the text. The TF-IDF representation only looked at _what_ words were present in a document, but not _where_ they were located and _in what order_.

Here we will begin with a very simple probabilistic language model and we will start with asking the question: what is the probability of a sentence and how can we estimate it from data (a text corpus)? 
Let's say that we want to score sentences based on how likely or unlikely they are to appear in a language (for example to detect non-sensical sentences).

:::{.ex-class}
**Alice was beginning to get very tired of sitting by her sister on the bank.**
:::

One simple approach could be to look at the frequency of each word in a large corpus of text and then multiply the frequencies of the words in the sentence to get an overall score. However, this is nothing more than the bag-of-words model and it ignores the order of the words. Another approach would be to treat the whole sentence as a single token and look up its frequency in a large corpus. However, you can imagine that most sentences will not appear in the corpus more than once and it will be difficult to estimate their probabilities.

A compromise between these two extremes is to look at sequences of several words and estimate the probabilities of these sequences and then combine them to get the probability of the whole sentence using the chain rule of probability.

Let's look at **Alice was beginning to get very tired** (shorter example to save space). Using the chain rule of probability, we can decompose the probability of the whole sentence into the product of conditional probabilities of each word given the previous words.

For events A and B, the probability that both occur can be written in terms of the conditional probability of A given B as

$$
P(A \cap B) = P(A|B) P(B)
$$

For three events A, B, and C, we can extend this to

:::{.ex-class}
$$
P(A \cap B \cap C) = P(A|B \cap C) P(B|C) P(C)
$$
:::

Applying this to our sentence, we get

$$
\begin{align*}
P(\text{Alice was beginning to get very tired}) & = P(\text{tired}|\text{Alice was beginning to get very})\\ 
& \times  P(\text{very}|\text{Alice was beginning to get}) \\
& \times P(\text{get}|\text{Alice was beginning to}) \\
& \times P(\text{to}|\text{Alice was beginning}) \\
& \times P(\text{beginning}|\text{Alice was}) \\
& \times P(\text{was}|\text{Alice}) \\
& \times P(\text{Alice})
\end{align*}
$$

Until now we have not made any assumptions about the dependencies between the words, we have simply restated the sentence probability using the chain rule. However, estimating these conditional probabilities is difficult because the number of possible combinations of words is very large and we will not have enough data to estimate them accurately.

We can get around this problem by making a Markov assumption: that the conditional probability of a word only depends on the previous word (this is called a first-order Markov assumption). With this assumption the probabilities above simplify to

$$
\begin{align*}
P(\text{Alice was beginning to get very tired}) & = P(\text{tired}|\text{very})\\ 
& \times  P(\text{very}|\text{get}) \\
& \times P(\text{get}|\text{to}) \\
& \times P(\text{to}|\text{beginning}) \\
& \times P(\text{beginning}|\text{was}) \\
& \times P(\text{was}|\text{Alice}) \\
& \times P(\text{Alice})
\end{align*}
$$

Now these pairs of consecutive words (called) bigrams are much easier to estimate from data. We only need to count how many times each word follows another word in a large corpus of text and divide by the total number of times the first word appears to get the conditional probability.

We could arrange the bigram probabilities in a matrix, where the rows correspond to the first word and the columns correspond to the second word. Each entry in the matrix would then contain the probability of the second word given the first word. This matrix is called a bigram model.

The estimate of a conditional probability is simply a ratio of counts,

:::{.ex-class}
$$
\hat{P}(w_t | w_{t-1}) = \frac{C(w_{t-1}, w_t)}{C(w_{t-1})}
$$
:::

where $C(\cdot)$ denotes a count in the corpus (this is the maximum likelihood estimate). For the tiny "corpus" consisting only of our one sentence the bigram matrix is:

|           | Alice | was | beginning | to  | get | very | tired |
|-----------|-------|-----|-----------|-----|-----|------|-------|
| Alice     | 0     | 1.0 | 0         | 0   | 0   | 0    | 0     |
| was       | 0     | 0   | 1.0       | 0   | 0   | 0    | 0     |
| beginning | 0     | 0   | 0         | 1.0 | 0   | 0    | 0     |
| to        | 0     | 0   | 0         | 0   | 1.0 | 0    | 0     |
| get       | 0     | 0   | 0         | 0   | 0   | 1.0  | 0     |
| very      | 0     | 0   | 0         | 0   | 0   | 0    | 1.0   |
| tired     | 0     | 0   | 0         | 0   | 0   | 0    | 0     |

The last row is empty because "tired" is never followed by another word in this tiny corpus. With a real corpus the table is much larger and its entries are less extreme. The last section of this chapter estimates the matrix from the whole book.

## Using the model: scoring and generating text

With the bigram probabilities at hand we can

- compute the probability of any sentence as $P(w_1) \prod_{t=2}^{T} P(w_t | w_{t-1})$ (in practice we add the logarithms of the probabilities, so that the product of many small numbers does not underflow), and
- generate text by drawing the first word, then repeatedly drawing the next word from the row of the matrix that belongs to the current word.

## Limitations: why counting is not enough

1. **Sparsity.** A vocabulary of $V$ words has $V^2$ possible bigrams (and $V^3$ trigrams, ...). Most of them never appear in a corpus, so their estimated probability is exactly zero, and every sentence that contains an unseen bigram gets probability zero, even if it is perfectly sensible. Smoothing methods (adding pseudo-counts, backing off to shorter contexts) patch this problem, but only partly.
2. **No notion of similarity.** To the counting model, the words "cat" and "dog" are as different as "cat" and "although": each word is a separate row of the table and seeing "the cat sat" tells us nothing about "the dog sat". In the one-hot representation all word vectors are orthogonal.
3. **Short memory.** Conditioning on one previous word ignores almost all of the context. Longer contexts (trigrams, ...) make the sparsity problem exponentially worse.

## The Bi-Gram Model as a Classification Problem

Instead of directly counting bi-grams in the text we can also obtain the same probabilities by posing the problem as a classification task. Given a word, we want to predict the next word in the sequence. We can treat the unique words in the corpus just as we treat categorical variables in a regression setting: we represent the categories (words) using indicator (dummy) variables. So to predict the word **was** given the word **Alice**, we would represent **Alice** as a (row) vector of zeros with a one in the position corresponding to **Alice** in the vocabulary and we will do the same for all other words. Let's say for a moment that we have a predictive model $f(x)$ that takes as input the indicator vector for a word and outputs a vector of predicted probabilities for the next word (the classes are the $V$ words of the vocabulary, and the true class is encoded by the one-hot vector $y$).

Let the vocabulary be {Alice, beginning, get, tired, to, very, was}. Then the training sample **Alice was** would be represented as

$$
f(x_{Alice}) = f\left(
\begin{bmatrix}
1 & 0 & 0 & 0 & 0 & 0 & 0
\end{bmatrix}
\right) =
\begin{bmatrix}
0.02 & 0.03 & 0.05 & 0.05 & 0.02 & 0.03 & 0.80
\end{bmatrix}
= \hat{y}_{Alice}
\quad y_{was} =
\begin{bmatrix}
0 & 0 & 0 & 0 & 0 & 0 & 1
\end{bmatrix}
$$

The predicted probabilities sum to one and the cross-entropy loss of this sample is $-\log 0.80$ (the loss is defined in the [logistic regression chapter](03-06-ANN-Logistic-Regression.ipynb)).




## The Bi-Gram Model as a Logistic Regression

We can now see how to implement the bi-gram model as a multi-class classification problem. We want to predict the second word in a sequence of two words. There are as many possibilities for the next word as there are words in the vocabulary, let's say $V$. In the simple example that follows we use a tiny piece of text with only five words so that it is easy to see what the model is doing.

The only thing that is special about this classification problem is that the input vectors have a special structure: they are indicator vectors with 1 at the position of the word in the vocabulary and 0 elsewhere. For example, the prediction of the next word after **Alice** would be done using the input vector $x_{Alice} = [1, 0, 0, 0, 0]$ (a row vector, in line with the notation of the following chapters):

$$
o = x_{Alice} W + b =
\underbrace{\begin{bmatrix}
1 & 0 & 0 & 0 & 0
\end{bmatrix}}_{x_{Alice}}
\begin{bmatrix}
w_{11} & w_{12} & w_{13} & w_{14} & w_{15} \\
w_{21} & w_{22} & w_{23} & w_{24} & w_{25} \\
w_{31} & w_{32} & w_{33} & w_{34} & w_{35} \\
w_{41} & w_{42} & w_{43} & w_{44} & w_{45} \\
w_{51} & w_{52} & w_{53} & w_{54} & w_{55} \\
\end{bmatrix}
+ \begin{bmatrix}
b_1 & b_2 & b_3 & b_4 & b_5
\end{bmatrix}
= \begin{bmatrix}
w_{11} + b_1 & w_{12} + b_2 & w_{13} + b_3 & w_{14} + b_4 & w_{15} + b_5
\end{bmatrix}
$$

Note that the matrix multiplication with the indicator vector simply selects the **row** of the weight matrix corresponding to the position of the 1 in the indicator vector. So in this case, we are selecting the first row of the weight matrix, because the position of **Alice** in the vocabulary is 1. The predicted probabilities for the next word are obtained with the softmax function:

:::{.ex-class}
$$
\hat{y}_k = \text{softmax}(o)_k = \frac{e^{o_k}}{\sum_{j=1}^V e^{o_j}}, \qquad k = 1, \ldots, V
$$
:::

Training the weights by gradient descent with the cross-entropy loss gives (up to the effect of the optimization) the same probabilities as counting. The [multi-class chapter](03-07-ANN-Multiclass-Classification.ipynb) trains this model.

## From counting to learning: the hidden layer

Learning the probabilities instead of counting them gives us flexibility: we can put a **hidden layer** with a small number $d \ll V$ of neurons between the input and the output,

:::{.ex-class}
$$
\hat{y} = \text{softmax}\big( \varphi(x W_h + b_h) W_o + b_o \big)
$$
:::

Now each word is first mapped to a short dense vector (the row of $W_h$ that belongs to the word, i.e. a *word embedding*), and the next-word probabilities are computed from this vector. Words that are followed by similar words end up with similar vectors, so statistical strength is shared across words and we need far fewer parameters ($2Vd$ instead of $V^2$). This is the idea behind the neural language models and the word embeddings that we study in this course.

To get there we need the building blocks of neural networks. The following chapters develop them one by one, from the simplest model to the bigram model as a neural network:

1. [Simple linear regression as a single neuron](03-03-ANN-Simple-Linear-Regression.ipynb): weights, bias, loss function and gradient descent.
2. [Simple regression with a hidden layer](03-04-ANN-Simple-Regression-Hidden-Layer.ipynb): activation functions, matrix notation and the gradients of a network with a hidden layer.
3. [Multiple regression with a hidden layer](03-05-ANN-Multiple-Regression.ipynb): several input features; multiple linear regression as a special case.
4. [Logistic regression with a hidden layer](03-06-ANN-Logistic-Regression.ipynb): entropy, the cross-entropy loss and classification with two classes.
5. [Multi-class classification](03-07-ANN-Multiclass-Classification.ipynb): the softmax function, its gradients, and finally the bigram model as a softmax network.

## A short history

The ideas in this chapter have a long history, which is worth knowing because the same questions keep returning in new forms.

- **Markov chains (1913).** Andrey Markov counted the vowels and consonants in the first 20,000 letters of Pushkin's *Eugene Onegin* and showed that whether a letter is a vowel or a consonant depends on the preceding letter [@MARKOV2006ExampleStatisticalInvestigation]. Such dependence of each element on the previous one is the first-order Markov assumption that we made above for words.
- **Information theory and n-grams (1948).** Claude Shannon introduced the entropy of a source and demonstrated how sequences of letters and words generated from n-gram statistics look more and more like English as $n$ grows [@SHANNON1948MathematicalTheoryCommunication]. The entropy and cross-entropy are the subject of the [logistic regression chapter](03-06-ANN-Logistic-Regression.ipynb).
- **Statistical language models (1980s to 2000s).** Word n-gram models with smoothing became the workhorse of speech recognition and machine translation; extensions such as class-based n-grams [@BROWN1992ClassBasedNgramModels] tackled the sparsity problem by grouping similar words. See @JURAFSKY2025SpeechLanguageProcessing for a textbook treatment.
- **Artificial neurons and learning (1943 to 1986).** The first mathematical model of a neuron [@MCCULLOCH1943LogicalCalculusIdeas], the perceptron with its learning rule [@ROSENBLATT1958PerceptronProbabilisticModel] and, importantly, the back-propagation algorithm for training networks with hidden layers [@RUMELHART1986LearningRepresentationsBackpropagating] are the foundations of the networks in the following chapters.
- **Neural language models (2003).** Bengio and co-authors combined the two lines of work: a neural network with a hidden layer learns word vectors and the next-word probabilities jointly, which overcomes the sparsity problem of n-gram models [@BENGIO2003NeuralProbabilisticLanguage].
- **Word embeddings and transformers (2013 and later).** Word2vec made it cheap to learn good word vectors from huge corpora [@MIKOLOV2013EfficientEstimationWord], and the transformer architecture [@VASWANI2017AttentionAllYou] replaced the fixed-size context by attention over the whole text.

## Implementation

We estimate the bigram probabilities from the complete text of *Alice's Adventures in Wonderland* (distributed with NLTK) by counting.

In [1]:
import re
from collections import Counter

import numpy as np
import pandas as pd
import nltk
from nltk.corpus import gutenberg

nltk.download('gutenberg', quiet=True)

alice_text = gutenberg.raw('carroll-alice.txt')

# A very simple tokenizer: lower-case words (letters and apostrophes only)
tokens = re.findall(r"[a-z']+", alice_text.lower())

print(f"Number of tokens: {len(tokens)}")
print(f"Vocabulary size: {len(set(tokens))}")
print("First tokens:", tokens[:12])

Number of tokens: 27802
Vocabulary size: 2856
First tokens: ["alice's", 'adventures', 'in', 'wonderland', 'by', 'lewis', 'carroll', 'chapter', 'i', 'down', 'the', 'rabbit']



First we reproduce the table from above: the corpus consists of a single sentence, so every row has at most one non-zero entry.

In [2]:
toy_sentence = "alice was beginning to get very tired".split()

toy_vocab = sorted(set(toy_sentence))
toy_index = {word: i for i, word in enumerate(toy_vocab)}

# Count how often each word (row) is followed by each other word (column)
toy_counts = np.zeros((len(toy_vocab), len(toy_vocab)))

for first_word, second_word in zip(toy_sentence[:-1], toy_sentence[1:]):
    toy_counts[toy_index[first_word], toy_index[second_word]] += 1

# Divide each row by its total (rows without any bigram stay zero)
row_totals = toy_counts.sum(axis=1, keepdims=True)
toy_probs = np.divide(toy_counts, row_totals, out=np.zeros_like(toy_counts), where=row_totals > 0)

pd.DataFrame(toy_probs, index=toy_vocab, columns=toy_vocab)

,alice,beginning,get,tired,to,very,was
alice,0.0,0.0,0.0,0.0,0.0,0.0,1.0
beginning,0.0,0.0,0.0,0.0,1.0,0.0,0.0
get,0.0,0.0,0.0,0.0,0.0,1.0,0.0
tired,0.0,0.0,0.0,0.0,0.0,0.0,0.0
to,0.0,0.0,1.0,0.0,0.0,0.0,0.0
very,0.0,0.0,0.0,1.0,0.0,0.0,0.0
was,0.0,1.0,0.0,0.0,0.0,0.0,0.0


### The whole book

Now we count all bigrams in the book. `first_counts[w]` is the number of times that `w` is followed by another word, which is the denominator $C(w_{t-1})$ of the estimator.

In [3]:
bigram_counts = Counter(zip(tokens[:-1], tokens[1:]))
first_counts = Counter(tokens[:-1])
unigram_counts = Counter(tokens)

def next_word_probs(word, top=5):
    """The `top` most likely next words and their estimated probabilities"""
    candidates = [(second, count / first_counts[word])
                  for (first, second), count in bigram_counts.items() if first == word]
    return sorted(candidates, key=lambda pair: -pair[1])[:top]

for word in ["alice", "the", "said"]:
    print(f"After '{word}':", [(w, round(p, 3)) for w, p in next_word_probs(word)])

After 'alice': [('was', 0.044), ('and', 0.031), ('thought', 0.031), ('had', 0.029), ('said', 0.029)]
After 'the': [('queen', 0.038), ('king', 0.037), ('mock', 0.033), ('gryphon', 0.033), ('hatter', 0.031)]
After 'said': [('the', 0.45), ('alice', 0.249), ('to', 0.084), ('in', 0.017), ('this', 0.013)]


### Scoring sentences

The log-probability of a sentence is the log of the probability of the first word plus the sum of the log bigram probabilities. A sentence with an unseen bigram gets probability zero (log-probability $-\infty$).

In [6]:
def sentence_log_prob(sentence):
    words = sentence.lower().split()
    log_prob = np.log(unigram_counts[words[0]] / len(tokens))

    for first, second in zip(words[:-1], words[1:]):
        if bigram_counts[(first, second)] == 0:
            return -np.inf     # unseen bigram: probability zero
        log_prob += np.log(bigram_counts[(first, second)] / first_counts[first])

    return log_prob

print("Natural order: ", sentence_log_prob("alice was beginning to get very tired"))
print("Reversed order:", sentence_log_prob("tired very get to beginning was alice"))

Natural order:  -23.67276080732909
Reversed order: -inf


### Generating text

We generate text by sampling the next word from the row of the (implicit) bigram matrix that belongs to the current word. The random number generator is seeded so that the output is reproducible.

In [5]:
rng = np.random.default_rng(1)

def sample_sentence(start_word, length=12):
    words = [start_word]
    for _ in range(length - 1):
        candidates = [(second, count) for (first, second), count in bigram_counts.items()
                      if first == words[-1]]
        if not candidates:
            break
        next_words, counts = zip(*candidates)
        words.append(rng.choice(next_words, p=np.array(counts) / sum(counts)))
    return " ".join(words)

for _ in range(3):
    print(sample_sentence("alice"))

alice come before she comes at all as it but it out
alice heard it put on shrinking rapidly she was the king 'and
alice said but come or two the door began ordering people began


### Sparsity

Finally, let's see how much of the $V \times V$ table we actually observe.

In [6]:
# no-ex-class
vocab_size = len(set(tokens))

print(f"Possible bigrams (V^2): {vocab_size ** 2:,}")
print(f"Observed bigrams: {len(bigram_counts):,}")
print(f"Share of the table that is observed: {len(bigram_counts) / vocab_size ** 2:.4%}")

Possible bigrams (V^2): 8,156,736
Observed bigrams: 15,071
Share of the table that is observed: 0.1848%
